In [ ]:
import numpy as np
import optuna
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler

from features import feature_engeneering, get_fe_parametrs
from folds import FOLDS, X, y


def objective_forest(trial: optuna.Trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 2500),
        'max_samples': trial.suggest_float('max_samples', 0.6, 1.0),
        'max_features': trial.suggest_float('max_features', 0.6, 1.0),
        'random_state': 131,
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'n_jobs': -1
    }

    aucs = []
    for tr_idx, val_idx in FOLDS:
        X_tr, _ = X.iloc[tr_idx], y.iloc[tr_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        qt, train_median, train_mad, tsh_q, fti_q, tt4_q = get_fe_parametrs(X_tr)

        X_tr_f = feature_engeneering(
            df=X_tr, qt=qt, train_median=train_median,
            train_mad=train_mad, tsh_quantile=tsh_q,
              fti_quantile=fti_q, tt4_quantile=tt4_q
        )
        
        X_val_f = feature_engeneering(
                    df=X_val, qt=qt, train_median=train_median,
                    train_mad=train_mad, tsh_quantile=tsh_q,
                      fti_quantile=fti_q, tt4_quantile=tt4_q
        )

        scaler = StandardScaler()
        scaler.fit(X_tr_f)

        X_tr_scaled = scaler.transform(X_tr_f)
        X_val_scaled = scaler.transform(X_val_f)

        model = IsolationForest(**params)
        model.fit(X_tr_scaled)

        val_scores = -model.decision_function(X_val_scaled)

        pr_auc = average_precision_score(y_val, val_scores)
        aucs.append(pr_auc)

    return np.mean(aucs)

study_forest = optuna.create_study(direction='maximize')
study_forest.optimize(objective_forest, n_trials=100)

print(f'Best_params: {study_forest.best_params}')
print(f'Best value: {study_forest.best_value}')

c:\Users\DNS\PycharmProjects\ds&ml\anomaly_and_pca\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-11-22 00:13:53,826] A new study created in memory with name: no-name-76049d30-07ee-4efd-85aa-b17ac6d051d0
[I 2026-11-22 00:13:59,679] Trial 0 finished with value: 0.31437343327674816 and parameters: {'n_estimators': 1411, 'max_samples': 0.9880838002562308, 'max_features': 0.7250714766313371, 'bootstrap': False}. Best is trial 0 with value: 0.31437343327674816.
[I 2026-11-22 00:14:16,317] Trial 1 finished with value: 0.30980276284459257 and parameters: {'n_estimators': 2106, 'max_samples': 0.9874222235689147, 'max_features': 0.9069314062622966, 'bootstrap': False}. Best is trial 0 with value: 0.31437343327674816.
[I 2026-11-22 00:14:26,066] Trial 2 finished with value: 0.32630862500823676 and parameters: 

Best_params: {'n_estimators': 562, 'max_samples': 0.7129247198337165, 'max_features': 0.6087634695737341, 'bootstrap': True}
Best value: 0.3473764766145583


In [2]:
best_params = study_forest.best_params
oof_forest = np.zeros(len(X))
synth_forest = []

for fold, (tr_idx, val_idx) in enumerate(FOLDS):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    qt, train_median, train_mad, tsh_q, fti_q, tt4_q = get_fe_parametrs(X_tr)

    X_tr_f = feature_engeneering(
        df=X_tr, qt=qt, train_median=train_median,
        train_mad=train_mad, tsh_quantile=tsh_q, 
        fti_quantile=fti_q, tt4_quantile=tt4_q 
    )
    X_val_f = feature_engeneering(
        df=X_val, qt=qt, train_median=train_median,
        train_mad=train_mad, tsh_quantile=tsh_q,
        fti_quantile=fti_q, tt4_quantile=tt4_q
    )

    X_tr_normal = X_tr_f[y_tr == 0]

    scaler = StandardScaler()
    scaler.fit(X_tr_normal)

    X_tr_scaled = scaler.transform(X_tr_normal)
    X_val_scaled = scaler.transform(X_val_f)

    X_synth = X_tr_normal.copy()
    for col in X_tr_normal.columns:
        X_synth[col] = np.random.permutation(X_synth[col].values)
    X_synth_scaled = scaler.transform(X_synth)


    model = IsolationForest(**best_params)
    model.fit(X_tr_scaled)

    oof_forest[val_idx] = -model.decision_function(X_val_scaled)
    synth_forest.extend(-model.decision_function(X_synth_scaled))

np.save('oof_forest.npy', oof_forest)
np.save('synth_forest.npy', synth_forest)